In [92]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn.utils.rnn as rnn_utils
from torch.nn.utils.rnn import pad_sequence


df = pd.read_pickle('task1_tokenized.pkl')

test_df = pd.read_pickle('test_tokensised.pkl')

ok = df.explode('labels')
unique_tags = set(ok['labels'].unique())
print(set(unique_tags))

tokens = df.explode('tokens')
unique_tokens = set(tokens['tokens'].unique())

UNK = "UNK"

word_to_idx = {word: idx for idx, word in enumerate(unique_tokens)}
word_to_idx[UNK] = len(word_to_idx)
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
idx_to_word[len(word_to_idx)-1] = UNK

# print(word_to_idx[UNK], idx_to_word[word_to_idx[UNK]])

# unique_tags = set(df['labels'].unique())
tag_to_idx = {tag: idx for idx, tag in enumerate(unique_tags)}
idx_to_tag = {idx: tag for tag, idx in tag_to_idx.items()}


{'O', 'I', 'B'}


In [93]:
class SentenceDataset(Dataset):
    def __init__(self, df):
        self.tokens = [torch.tensor([word_to_idx.get(word, word_to_idx[UNK]) for word in tokens]) for tokens in df['tokens'].tolist()]
        self.labels = [torch.tensor([tag_to_idx[tag] for tag in labels]) for labels in df['labels'].tolist()]

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        return self.tokens[idx], self.labels[idx]

def collate_fn(batch):
    tokens, labels = zip(*batch)
    padded_tokens = pad_sequence(tokens, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=0)
    return padded_tokens, padded_labels
EMBEDDING_DIM = 200
HIDDEN_DIM = 128
NUM_LAYERS = 2
BATCH_SIZE = 16
NUM_EPOCHS = 5

dataset = SentenceDataset(df)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)


test_dataset = SentenceDataset(test_df)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        logits = self.fc(output)
        return logits

vocab_size = len(unique_tokens) + 1 # For UNK
output_dim = len(unique_tags)
model = BiLSTMClassifier(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, output_dim, NUM_LAYERS)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

for epoch in range(NUM_EPOCHS):
    for tokens, labels in dataloader:
        optimizer.zero_grad()
        logits = model(tokens)
        logits_flattened = logits.view(-1, logits.shape[-1])  # Flatten logits
        labels_flattened = labels.view(-1)  # Flatten labels
        loss = criterion(logits_flattened, labels_flattened)
        loss.backward()
        optimizer.step()


        # print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {loss.item():.4f}")

    model.eval()
    total_pred = []
    total_true = []
    with torch.no_grad():
        correct = 0
        total = 0
        for tokens, labels in test_dataloader:
            logits = model(tokens)
            predictions = torch.argmax(logits, dim=2)

            total_pred.append(predictions.view(-1).tolist())
            total_true.append(labels.view(-1).tolist())
            for pred, label in zip(predictions, labels):
                correct += sum(pred == torch.tensor(label)).item()
                total += len(label)
        accuracy = correct / total
        print(f"Accuracy: {accuracy:.4f}")

corr = 0
wrong = 0


for idx, (pred, true_label) in enumerate(zip(total_pred, total_true)):
    # checking only for keyphrases, not O
    # print(pred)
    # print(true_label)
    # print()

    for i in range(len(pred)):
        if pred[i] == 1:
            if i == 0:
                total_pred[idx][i] = 2
            else:
                if pred[i-1] == 0:
                    total_pred[idx][i] = 2
        if (true_label[i] != 0) and (pred[i] == true_label[i]):
            corr += 1
        elif (true_label[i] != 0) and (pred[i] != true_label[i]):
            wrong += 1


    # if pred == true_label and true_label != 0:
    #     corr += 1
    # elif pred != true_label and true_label != 0:
    #     wrong += 1
    # print(f"Index: {idx}, Prediction: {pred}, True Label: {true_label}")
# for idx, (pred, true_label) in enumerate(zip(total_pred, total_true)):
#     print(pred)
#     print(true_label)
#     print()




<ipython-input-93-2e565310cd32>:82: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  correct += sum(pred == torch.tensor(label)).item()


Accuracy: 0.8318
Accuracy: 0.8324
Accuracy: 0.8170
Accuracy: 0.8234
Accuracy: 0.8292


In [94]:
print(corr/(corr+wrong))

print(idx_to_tag[2])

0.445692025664528
B


In [95]:
from sklearn.metrics import precision_recall_fscore_support

flattened_total_predicted = [item for sublist in total_pred for item in sublist]
flattened_total_true = [item for sublist in total_true for item in sublist]

precision, recall, f1, _ = precision_recall_fscore_support(flattened_total_true, flattened_total_predicted, average=None)

print("Class-wise Precision:", precision)
print("Class-wise Recall:", recall)
print("Class-wise F1-score:", f1)


Class-wise Precision: [0.9116329  0.44810027 0.40763359]
Class-wise Recall: [0.90512607 0.45057109 0.43890411]
Class-wise F1-score: [0.90836783 0.44933229 0.42269129]


In [76]:
torch.save(model.state_dict(), 'bilstm_model.pth')

In [83]:
modill = BiLSTMClassifier(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, output_dim, NUM_LAYERS)
modill.load_state_dict(torch.load('bilstm_model.pth'))
modill.eval()

BiLSTMClassifier(
  (embedding): Embedding(11564, 200)
  (lstm): LSTM(200, 128, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)

In [84]:
modill.eval()
total_pred = []
total_true = []
with torch.no_grad():
    correct = 0
    total = 0
    for tokens, labels in test_dataloader:
        logits = modill(tokens)
        predictions = torch.argmax(logits, dim=2)

        total_pred.append(predictions.view(-1).tolist())
        total_true.append(labels.view(-1).tolist())
        for pred, label in zip(predictions, labels):
            correct += sum(pred == torch.tensor(label)).item()
            total += len(label)
    accuracy = correct / total
    print(f"Accuracy: {accuracy:.4f}")


<ipython-input-84-ed2330455dd4>:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  correct += sum(pred == torch.tensor(label)).item()


Accuracy: 0.7737


In [85]:

corr = 0
wrong = 0


for idx, (pred, true_label) in enumerate(zip(total_pred, total_true)):
    # checking only for keyphrases, not O
    # print(pred)
    # print(true_label)
    # print()

    for i in range(len(pred)):
        if pred[i] == 1:
            if i == 0:
                total_pred[idx][i] = 2
            else:
                if pred[i-1] == 0:
                    total_pred[idx][i] = 2
        if (true_label[i] != 0) and (pred[i] == true_label[i]):
            corr += 1
        elif (true_label[i] != 0) and (pred[i] != true_label[i]):
            wrong += 1


    # if pred == true_label and true_label != 0:
    #     corr += 1
    # elif pred != true_label and true_label != 0:
    #     wrong += 1
    # print(f"Index: {idx}, Prediction: {pred}, True Label: {true_label}")
# for idx, (pred, true_label) in enumerate(zip(total_pred, total_true)):
#     print(pred)
#     print(true_label)
#     print()

In [86]:

print(corr/(corr+wrong))

from sklearn.metrics import precision_recall_fscore_support

flattened_total_predicted = [item for sublist in total_pred for item in sublist]
flattened_total_true = [item for sublist in total_true for item in sublist]

# Calculate precision, recall, and F1-score
precision, recall, f1, _ = precision_recall_fscore_support(flattened_total_true, flattened_total_predicted, average=None)

print("Class-wise Precision:", precision)
print("Class-wise Recall:", recall)
print("Class-wise F1-score:", f1)

0.5582034830430798
Class-wise Precision: [0.93897357 0.34882701 0.34917355]
Class-wise Recall: [0.82508343 0.62662466 0.4630137 ]
Class-wise F1-score: [0.87835204 0.44816901 0.39811543]


Class-wise Precision: [0.93897357 0.34882701 0.34917355]
Class-wise Recall: [0.82508343 0.62662466 0.4630137 ]
Class-wise F1-score: [0.87835204 0.44816901 0.39811543]

Class-wise Precision: [0.91766243 0.41523263 0.36227123]
Class-wise Recall: [0.8791713  0.51319417 0.4230137 ]
Class-wise F1-score: [0.89800459 0.45904527 0.39029323]

In [87]:
Class-wise Precision: [0.93897357 0.34882701 0.34917355]
Class-wise Recall: [0.82508343 0.62662466 0.4630137 ]
Class-wise F1-score: [0.87835204 0.44816901 0.39811543]

Class-wise Precision: [0.91766243 0.41523263 0.36227123]
Class-wise Recall: [0.8791713  0.51319417 0.4230137 ]
Class-wise F1-score: [0.89800459 0.45904527 0.39029323]

SyntaxError: invalid syntax (<ipython-input-87-289bb9e88118>, line 1)